In [1]:
import json
import joblib
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

In [2]:
PLANT = 1
OUT = Path(r"C:\Users\VICTUS\Documents\College\Semester 3\Machine Learning\Solar Power Generation Data\processed")
 
X = pd.read_csv(OUT / f"plant{PLANT}_features.csv", index_col="DATE_TIME",
                parse_dates=["DATE_TIME", "target_time"])
sets = json.load(open(OUT / "feature_sets.json"))

In [3]:
MODELS = {
    "RF": (RandomForestRegressor(random_state=42, n_jobs=-1),
           {"n_estimators": [300], "max_depth": [8, 16, None], "min_samples_leaf": [1, 5]}),
    "GB": (HistGradientBoostingRegressor(random_state=42),
           {"learning_rate": [0.03, 0.1], "max_iter": [200, 400], "max_depth": [3, 6]}),
}
cv = TimeSeriesSplit(n_splits=4, gap=4)   # chronological folds, never shuffled
 
preds, best = [], {}
for setup, feats in sets.items():
    d = X[X["split"].isin(["train", "test"])].dropna(subset=feats + ["y"]).sort_index()
    tr, te = d[d["split"] == "train"], d[d["split"] == "test"]
    for name, (est, grid) in MODELS.items():
        gs = GridSearchCV(est, grid, cv=cv, scoring="neg_root_mean_squared_error", n_jobs=1)
        gs.fit(tr[feats], tr["y"])             # tuning sees TRAIN rows only
        print(f"{setup:16s} {name}: CV RMSE {-gs.best_score_:8.1f}  params {gs.best_params_}")
        best[f"{setup}_{name}"] = {"params": gs.best_params_, "cv_rmse": -gs.best_score_}
        joblib.dump(gs.best_estimator_, OUT / f"model_{setup}_{name}.joblib")
        preds.append(pd.DataFrame({
            "target_time": te["target_time"], "setup": setup, "model": name,
            "y_true": te["y"], "y_pred": gs.best_estimator_.predict(te[feats])}))
 
pd.concat(preds).to_csv(OUT / f"plant{PLANT}_predictions.csv")
with open(OUT / "best_params.json", "w") as f:
    json.dump(best, f, indent=2, default=str)
print(f"\nSaved predictions, models and best_params.json to {OUT}")

A_lagged_only    RF: CV RMSE   3886.5  params {'max_depth': 8, 'min_samples_leaf': 1, 'n_estimators': 300}


c:\Users\VICTUS\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\VICTUS\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\VICTUS\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\VICTUS\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\VICTUS\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.Crea

A_lagged_only    GB: CV RMSE   3965.3  params {'learning_rate': 0.03, 'max_depth': 3, 'max_iter': 200}
B_with_forecast  RF: CV RMSE    959.8  params {'max_depth': 8, 'min_samples_leaf': 5, 'n_estimators': 300}
B_with_forecast  GB: CV RMSE    990.4  params {'learning_rate': 0.03, 'max_depth': 3, 'max_iter': 200}

Saved predictions, models and best_params.json to C:\Users\VICTUS\Documents\College\Semester 3\Machine Learning\Solar Power Generation Data\processed
